In [1]:
from pathlib import Path

csv_files = list(Path("../data/raw").glob("*.csv"))

for csv_file in csv_files:
    print(csv_file.name)

Womens Clothing E-Commerce Reviews.csv


In [2]:
import pandas as pd

reviews_data = pd.read_csv(
    "../data/raw/Womens Clothing E-Commerce Reviews.csv"
)

print("Rows and columns:", reviews_data.shape)
print(reviews_data.columns.tolist())

Rows and columns: (23486, 11)
['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


In [3]:
reviews_data = reviews_data.dropna(subset=["Review Text", "Rating"]).copy()

reviews_data["sentiment"] = reviews_data["Rating"].map(
    {
        1: "negative",
        2: "negative",
        3: "neutral",
        4: "positive",
        5: "positive",
    }
)

reviews_data = reviews_data[
    ["Review Text", "sentiment"]
].rename(
    columns={"Review Text": "text"}
)

print(reviews_data["sentiment"].value_counts())

sentiment
positive    17448
neutral      2823
negative     2370
Name: count, dtype: int64


In [4]:
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

reviews_data["cleaned_text"] = reviews_data["text"].apply(clean_text)

x_train, x_test, y_train, y_test = train_test_split(
    reviews_data["cleaned_text"],
    reviews_data["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=reviews_data["sentiment"],
)

sentiment_model = Pipeline([
    (
        "vectorizer",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
        ),
    ),
])

sentiment_model.fit(x_train, y_train)

test_accuracy = sentiment_model.score(x_test, y_test)

print("Real dataset test accuracy:", round(test_accuracy * 100, 2), "%")

Real dataset test accuracy: 80.3 %


In [5]:
import joblib
from sklearn.metrics import classification_report

predictions = sentiment_model.predict(x_test)

print(
    classification_report(
        y_test,
        predictions,
        digits=3,
    )
)

sentiment_model.fit(
    reviews_data["cleaned_text"],
    reviews_data["sentiment"],
)

joblib.dump(
    sentiment_model,
    "../app/models/sentiment_model.pkl",
)

print("Final real-data sentiment model saved.")

              precision    recall  f1-score   support

    negative      0.505     0.591     0.545       474
     neutral      0.381     0.512     0.437       565
    positive      0.954     0.879     0.915      3490

    accuracy                          0.803      4529
   macro avg      0.613     0.660     0.632      4529
weighted avg      0.835     0.803     0.817      4529

Final real-data sentiment model saved.
